# AuctionChain Vehicle Price Prediction - Exploratory Analysis

This notebook is a lightweight, reproducible companion to the training pipeline. It inspects the raw US-market dataset and the processed feature frame. The authoritative metrics and decisions live in `reports/evaluation_report.txt` and `models/model_metadata.json`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
RAW_PATH = PROJECT_ROOT / "dataset" / "raw" / "used_cars.csv"
FEATURES_PATH = PROJECT_ROOT / "dataset" / "processed" / "features.csv"

sns.set_theme(style="whitegrid")
print(PROJECT_ROOT)

In [ ]:
raw = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False)
raw = raw.replace({"": pd.NA})
print(raw.shape)
raw.head()

In [ ]:
missing = raw.isna().sum().sort_values(ascending=False)
missing[missing > 0].to_frame("missing_rows")

In [ ]:
from src.data.load_dataset import parse_mileage, parse_price

prices = parse_price(raw["price"])
mileage = parse_mileage(raw["milage"])
years = pd.to_numeric(raw["model_year"], errors="coerce")

summary = pd.DataFrame(
    {
        "price_usd": prices.describe(),
        "mileage_mi": mileage.describe(),
        "model_year": years.describe(),
    }
)
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(prices, bins=60)
axes[0].set(title="Price distribution (USD)", xlabel="Price", ylabel="Listings")
axes[1].hist(mileage, bins=60)
axes[1].set(title="Mileage distribution (miles)", xlabel="Mileage", ylabel="Listings")
plt.tight_layout()

In [ ]:
brand_counts = raw["brand"].value_counts().head(15)
ax = brand_counts.plot(kind="barh", figsize=(8, 5), title="Top 15 brands by listing count")
ax.invert_yaxis()
ax.set_xlabel("Listings")
plt.tight_layout()

In [ ]:
features = pd.read_csv(FEATURES_PATH)
print(features.shape)
features[["brand", "model", "model_year", "mileage", "vehicle_age", "price", "split"]].head()

In [ ]:
numeric_columns = [
    "price",
    "mileage",
    "vehicle_age",
    "mileage_per_year",
    "horsepower",
    "displacement_l",
    "cylinders",
    "transmission_speeds",
]
correlation = features[numeric_columns].corr(numeric_only=True)
plt.figure(figsize=(9, 7))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Numeric feature correlation")
plt.tight_layout()

In [ ]:
features.groupby("split")["price"].agg(["count", "mean", "median", "max"]).round(2)

## Key conclusions

- Prices are USD and mileage is miles; both are heavily right-skewed, so MAE is used as the training objective.
- `clean_title` has no usable variance and is excluded from features.
- The split is grouped by vehicle signature to prevent duplicate-listing leakage.
- The final LightGBM model is evaluated once on the isolated test set; see `reports/evaluation_report.txt` for measured metrics and caveats.